# 🚗 GREENSM VEHICLE DETECTION - END-TO-END DATA PIPELINE
### Quy trình 2 Giai đoạn: Intra-Video FastDedup (pHash + Keep-Most-Vehicles) & Automated Data Profiling

**Dự án**: GreenSM Traffic AI (5 Classes: `greensm`, `car`, `motorcycle`, `bus`, `truck`)

**Quy trình chuẩn hóa gồm 10 bước:**
1. ⚙️ Cài đặt môi trường & Kiểm tra GPU
2. 📁 Tạo cấu trúc thư mục dữ liệu
3. 🏷️ Cấu hình Taxonomy 5 Classes & Quy tắc đặt tên chuẩn (Không dấu cách, Group Video ID)
4. 🌊 Trình đọc Streamer tiết kiệm RAM (1 FPS Video & EXIF Image Generator)
5. 🔬 **Stage 1**: Intra-Video FastDedup (`pHash` + `Keep-Most-Vehicles`, Hamming Distance < 5)
6. 📊 **Stage 2**: Automated Profiling (Day/Night robust, Blur, Occlusion IoA, Small Object, Pre-labels)
7. 🚀 Thực thi toàn bộ Pipeline trên tập Video & Ảnh
8. 📈 Trực quan hóa thống kê phân bố Metadata
9. 📦 Đóng gói chuẩn CVAT 1.1 (Native XML + Batch 50 frames/zip)
10. 💾 Tải gói dữ liệu và báo cáo về máy tính


## ⚙️ Bước 1: Kiểm tra phần cứng GPU & Cài đặt Thư viện
Chạy cell dưới đây để cài đặt Ultralytics (YOLO11), ImageHash, OpenCV và các thư viện cần thiết.

In [ ]:
!nvidia-smi
!pip install -q ultralytics imagehash opencv-python pillow tqdm pandas matplotlib seaborn

## 📁 Bước 2: Khởi tạo Cấu trúc Thư mục Dự án
Tạo cây thư mục chuẩn để chứa dữ liệu đầu vào và đầu ra.

In [ ]:
import os

DIRS = [
    'data/inputs/videos',
    'data/inputs/images',
    'data/processed_frames',
    'data/cvat_ready',
    'data/reports'
]

for d in DIRS:
    os.makedirs(d, exist_ok=True)
    print(f'📁 Đã tạo thư mục: {d}')
print('✅ Cấu trúc thư mục hoàn tất!')

### 📥 Hướng dẫn Đưa Dữ liệu vào Colab:
- **Cách 1 (Kéo thả)**: Mở thanh Files bên trái Colab, vào `data/inputs/videos` hoặc `data/inputs/images` kéo thả file vào.
- **Cách 2 (Google Drive)**: Mount Google Drive để copy dữ liệu vào (chạy cell tùy chọn bên dưới).

In [ ]:
# [TÙY CHỌN] Chạy cell này nếu muốn nạp dữ liệu từ Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/my_videos/*.mp4 data/inputs/videos/

## 🏷️ Bước 3: Định nghĩa Cấu hình, Taxonomy 5 Classes & Đặt tên Chuẩn
Quy tắc đặt tên:
- Video frames: `vid_<random_id_video>_<frame_stt>.jpg` (vd: `vid_v8a1_000001.jpg`)
- Ảnh đơn: `img_<random_8_ky_tu>.jpg` (vd: `img_a4f89c2e.jpg`)

In [ ]:
# ==============================================================================
# BƯỚC 3: CẤU HÌNH HỆ THỐNG & ĐẶT TÊN CHUẨN (Taxonomy & Utilities)
# ==============================================================================

"""
Configuration and Taxonomy for GreenSM Vehicle Detection Pipeline
"""

from pathlib import Path

# GreenSM 5-Class Label Taxonomy (Mentor-Aligned)
# 0: greensm (VinFast electric taxis/rental cars - cyan/logo)
# 1: car (general passenger cars, conventional taxis)
# 2: motorcycle (all motorcycles/scooters)
# 3: bus (city buses, coaches)
# 4: truck (trucks, pickups, heavy vehicles)
GREENSM_CLASSES = [
    "greensm",
    "car",
    "motorcycle",
    "bus",
    "truck"
]

CLASS_COLORS = {
    "greensm": "#00FFFF",  # Cyan
    "car": "#FF0000",      # Red
    "motorcycle": "#00FF00", # Green
    "bus": "#9C27B0",      # Purple
    "truck": "#FF9800"     # Orange
}

# COCO Dataset Vehicle IDs (YOLO11n COCO pretrained)
# 2: car, 3: motorcycle, 5: bus, 7: truck
COCO_VEHICLE_CLASS_IDS = {2, 3, 5, 7}

# Mapping COCO Vehicle IDs to GreenSM 5-Class IDs
# Note: Class 0 (greensm) is reserved for human annotators on CVAT
COCO_TO_GREENSM_MAP = {
    2: 1,  # car -> 1
    3: 2,  # motorcycle -> 2
    5: 3,  # bus -> 3
    7: 4   # truck -> 4
}

# Supported file extensions
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.flv', '.wmv', '.webm'}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff', '.tif'}

# Default Pipeline Thresholds
DEFAULT_HD_THRESH = 5            # Hamming distance for pHash duplicate clustering
DEFAULT_CONF_THRESH = 0.25        # YOLO detection confidence threshold
DEFAULT_MODEL_PATH = "yolo11n.pt" # Pretrained model weights
DEFAULT_BATCH_SIZE = 50          # Number of images per CVAT zip batch

# Data Profiling Thresholds (Stage 2)
NIGHT_V_THRESH = 65.0            # HSV V-channel mean threshold
NIGHT_P90_THRESH = 100.0         # 90th percentile ambient light threshold
NIGHT_EXTREME_THRESH = 50.0      # Extreme darkness threshold
BLUR_THRESH = 100.0              # Laplacian variance threshold
SMALL_OBJECT_RATIO = 0.01        # BBox area / Image area < 1%
OCCLUSION_IOA_THRESH = 0.30      # Intersection over Area > 30%
DENSE_SCENE_COUNT = 12           # Vehicle count > 12

# Project Directory Defaults
DEFAULT_INPUT_DIR = Path("data/inputs")
DEFAULT_VIDEO_DIR = DEFAULT_INPUT_DIR / "videos"
DEFAULT_IMAGE_DIR = DEFAULT_INPUT_DIR / "images"
DEFAULT_OUTPUT_DIR = Path("data/outputs")


"""
Utility functions: Standard Naming, Hashing, and Geometry Calculations
"""

import re
import hashlib
import uuid
from pathlib import Path


def sanitize_slug(name):
    """
    Sanitize text to a clean slug:
    Lowercase, alphanumeric and underscores only.
    """
    clean = re.sub(r'[^a-zA-Z0-9]+', '_', str(name)).strip('_').lower()
    return clean


def generate_image_id(identifier_str=None):
    """
    Generate an 8-character hex ID string (e.g. 'a4f89c2e').
    Uses MD5 of identifier if provided (deterministic), else random hex.
    """
    if identifier_str:
        return hashlib.md5(str(identifier_str).encode('utf-8')).hexdigest()[:8]
    return uuid.uuid4().hex[:8]


def generate_video_id(video_name):
    """
    Generate a 4-character ID for grouping frames from the same video (e.g. 'v8a1').
    """
    h = hashlib.md5(str(video_name).encode('utf-8')).hexdigest()[:4]
    return f"v{h[:3]}" if not h[0].isalpha() else h


def get_standard_image_name(source_type, source_name, index, original_stem=None):
    """
    Standard project-wide naming rule:
    - Image: img_<random_8_chars>.jpg (e.g. img_a4f89c2e.jpg)
    - Video: vid_<video_id>_<frame_stt:06d>.jpg (e.g. vid_v8a1_000001.jpg)
    """
    if source_type == "vid":
        vid_code = generate_video_id(source_name)
        return f"vid_{vid_code}_{index:06d}.jpg"
    else:
        key = original_stem if original_stem else f"{source_name}_{index}"
        img_code = generate_image_id(key)
        return f"img_{img_code}.jpg"


def calculate_ioa(box_a_xyxy, box_b_xyxy):
    """
    Calculate Intersection over Area (IoA) for box B covered by box A:
    IoA = Intersection_Area(box_A, box_B) / Area(box_B)
    """
    x1_a, y1_a, x2_a, y2_a = box_a_xyxy
    x1_b, y1_b, x2_b, y2_b = box_b_xyxy

    inter_x1 = max(x1_a, x1_b)
    inter_y1 = max(y1_a, y1_b)
    inter_x2 = min(x2_a, x2_b)
    inter_y2 = min(y2_a, y2_b)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_b = (x2_b - x1_b) * (y2_b - y1_b)
    if area_b <= 0:
        return 0.0

    return inter_area / area_b


print('✅ Cấu hình hệ thống & Hàm đặt tên chuẩn đã sẵn sàng!')


## 🌊 Bước 4: Bộ đọc Dữ liệu Generator Streamer (Chống tràn RAM)
Đọc video đúng tần số 1 FPS và đọc ảnh giữ nguyên orientation EXIF.

In [ ]:
# ==============================================================================
# BƯỚC 4: BỘ ĐỌC DỮ LIỆU STREAMER (RAM-Efficient Generators)
# ==============================================================================

"""
Data Streamers: Memory-efficient Generator Streamers for Video and Images
"""

import os
from pathlib import Path
from datetime import datetime
import cv2
import numpy as np
from PIL import Image

from .config import (
    VIDEO_EXTENSIONS,
    IMAGE_EXTENSIONS,
    DEFAULT_INPUT_DIR,
    DEFAULT_VIDEO_DIR,
    DEFAULT_IMAGE_DIR,
)
from .utils import get_standard_image_name


def get_exif_timestamp(img_path):
    """Attempt to extract EXIF timestamp from an image file."""
    try:
        pil_img = Image.open(img_path)
        exif = pil_img._getexif()
        if exif:
            for tag_id in (36867, 306):
                if tag_id in exif:
                    dt_str = exif[tag_id]
                    dt = datetime.strptime(dt_str, "%Y:%m:%d %H:%M:%S")
                    return dt.timestamp()
    except Exception:
        pass
    return os.path.getmtime(img_path)


def video_frame_streamer(video_path):
    """
    Generator streaming 1 frame per second (1 FPS) from a video file to avoid RAM overflow.
    Uses standardized naming: vid_<video_id>_<idx:06d>.jpg (NO SPACES).
    Yields: dict with keys: 'frame_id', 'timestamp', 'bgr_img', 'name', 'source'
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video file: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0 or np.isnan(fps):
        fps = 30.0

    frame_interval = max(1, int(round(fps)))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    video_name = Path(video_path).name
    current_frame_idx = 0
    sampled_count = 0

    print(f"[INFO] Video loaded: {video_name} | FPS: {fps:.2f} | Total Frames: {total_frames} | Frame Step (1 FPS): {frame_interval}")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if current_frame_idx % frame_interval == 0:
            sampled_count += 1
            sec = current_frame_idx / fps
            frame_name = get_standard_image_name("vid", video_name, sampled_count)
            yield {
                "frame_id": current_frame_idx,
                "timestamp": sec,
                "bgr_img": frame,
                "name": frame_name,
                "source": video_name
            }

        current_frame_idx += 1

    cap.release()


def image_dir_streamer(img_dir_path):
    """
    Generator streaming images sorted by EXIF timestamp / file mtime.
    Uses standardized naming: img_<random_8_chars>.jpg (NO SPACES).
    Yields: dict with keys: 'frame_id', 'timestamp', 'bgr_img', 'name', 'source'
    """
    path_obj = Path(img_dir_path)
    img_files = [
        f for f in path_obj.iterdir()
        if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
    ]

    if not img_files:
        raise RuntimeError(f"No valid image files found in directory: {img_dir_path}")

    print(f"[INFO] Reading {len(img_files)} images from directory: {path_obj.name}")
    print("[INFO] Extracting EXIF/timestamps for sorting & time grouping...")

    img_with_time = []
    for img_path in img_files:
        ts = get_exif_timestamp(img_path)
        img_with_time.append((img_path, ts))

    img_with_time.sort(key=lambda x: (x[1], x[0].name))

    for idx, (img_path, ts) in enumerate(img_with_time, start=1):
        bgr_img = cv2.imread(str(img_path))
        if bgr_img is None:
            continue

        std_name = get_standard_image_name("img", path_obj.name, idx, original_stem=img_path.stem)

        yield {
            "frame_id": idx,
            "timestamp": ts,
            "bgr_img": bgr_img,
            "name": std_name,
            "source": path_obj.name
        }


def auto_detect_inputs():
    """Auto-detect all inputs from data/inputs/videos or data/inputs/images."""
    inputs = []

    # Check for videos in data/inputs/videos
    if DEFAULT_VIDEO_DIR.exists():
        video_files = sorted([
            f for f in DEFAULT_VIDEO_DIR.iterdir()
            if f.is_file() and f.suffix.lower() in VIDEO_EXTENSIONS
        ])
        inputs.extend(video_files)

    # Check for images in data/inputs/images
    if DEFAULT_IMAGE_DIR.exists():
        image_files = [
            f for f in DEFAULT_IMAGE_DIR.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
        ]
        if image_files:
            inputs.append(DEFAULT_IMAGE_DIR)

    # Check root data/inputs directory as fallback
    if DEFAULT_INPUT_DIR.exists():
        root_videos = sorted([
            f for f in DEFAULT_INPUT_DIR.iterdir()
            if f.is_file() and f.suffix.lower() in VIDEO_EXTENSIONS
        ])
        inputs.extend(root_videos)

    return inputs


print('✅ Streamers đã sẵn sàng!')


## 🔬 Bước 5: Giai đoạn 1 — Intra-Video FastDedup
**Thuật toán của Lead**: `pHash + Keep-Most-Vehicles` (Hamming Distance < 5).
Khi 2 frame liên tiếp có visual tương đồng (`pHash < 5`), so sánh mật độ xe phát hiện bởi YOLO11n. Giữ lại frame có mật độ xe cao nhất!

In [ ]:
# ==============================================================================
# BƯỚC 5: STAGE 1 — INTRA-VIDEO FASTDEDUP (pHash + Keep-Most-Vehicles, HD < 5)
# Thuật toán Perceptual Hashing with Object-Density Selection
# ==============================================================================

"""
Stage 1: Intra-Video FastDedup
Algorithm: Perceptual Hashing with Object-Density Selection (pHash + Keep-Most-Vehicles, HD < 5)
"""

import cv2
from PIL import Image
import imagehash
from tqdm import tqdm

from .config import COCO_VEHICLE_CLASS_IDS, DEFAULT_HD_THRESH, DEFAULT_CONF_THRESH


def compute_phash(bgr_img):
    """Compute Perceptual Hash (pHash) of an OpenCV BGR image using PIL & imagehash."""
    rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb_img)
    return imagehash.phash(pil_img)


def run_yolo_on_frame(model, frame_item, conf_thresh=DEFAULT_CONF_THRESH, device=""):
    """
    Run YOLO11n inference on a single frame and extract vehicle detections.
    Returns: list of dicts [{cls, conf, bbox_xywhn, bbox_xyxy}]
    """
    results = model.predict(
        source=frame_item["bgr_img"],
        conf=conf_thresh,
        device=device if device else None,
        verbose=False
    )

    detections = []
    if not results or len(results) == 0:
        return detections

    res = results[0]

    if res.boxes is not None and len(res.boxes) > 0:
        for box in res.boxes:
            cls_id = int(box.cls[0].item())
            conf = float(box.conf[0].item())

            # Filter for COCO vehicle classes: car(2), motorcycle(3), bus(5), truck(7)
            if cls_id in COCO_VEHICLE_CLASS_IDS:
                xywhn = box.xywhn[0].cpu().numpy().tolist()
                xyxy = box.xyxy[0].cpu().numpy().tolist()

                detections.append({
                    "cls": cls_id,
                    "conf": conf,
                    "bbox_xywhn": xywhn,
                    "bbox_xyxy": xyxy
                })

    return detections


def resolve_cluster_keep_most_vehicles(cluster_items, model, conf_thresh=DEFAULT_CONF_THRESH, device=""):
    """
    Keep-Most-Vehicles Conflict Resolution Rule (Object-Density Selection):
    For a cluster of duplicate frames (HD < threshold), run YOLO11n to count vehicles.
    Keep the SINGLE frame with the MAX vehicle count (tie-broken by Laplacian variance sharpness).
    Returns: winning frame item with cached YOLO detections.
    """
    best_item = None
    max_count = -1
    best_sharpness = -1.0

    for item in cluster_items:
        if "detections" not in item:
            item["detections"] = run_yolo_on_frame(model, item, conf_thresh, device)

        vehicle_count = len(item["detections"])

        gray = cv2.cvtColor(item["bgr_img"], cv2.COLOR_BGR2GRAY)
        sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        item["laplacian_var"] = sharpness

        if (vehicle_count > max_count) or (vehicle_count == max_count and sharpness > best_sharpness):
            max_count = vehicle_count
            best_sharpness = sharpness
            best_item = item

    return best_item


def run_fast_dedup(streamer, model, hd_thresh=DEFAULT_HD_THRESH, conf_thresh=DEFAULT_CONF_THRESH, device="", is_image_dir=False, desc="Deduplicating"):
    """
    Run Stage 1 Dual-Stream Fast Deduplication on a frame generator.
    Returns: (survived_items, total_processed_frames, total_clusters)
    """
    current_cluster = []
    current_root_hash = None
    current_root_ts = None

    survived_items = []
    total_processed_frames = 0
    total_clusters = 0

    def process_and_finalize_cluster(cluster_list):
        nonlocal total_clusters
        if not cluster_list:
            return
        total_clusters += 1
        winner = resolve_cluster_keep_most_vehicles(cluster_list, model, conf_thresh, device)
        survived_items.append(winner)

    for frame_item in tqdm(streamer, desc=desc):
        total_processed_frames += 1
        frame_hash = compute_phash(frame_item["bgr_img"])
        frame_item["phash"] = frame_hash

        if not current_cluster:
            current_cluster = [frame_item]
            current_root_hash = frame_hash
            current_root_ts = frame_item["timestamp"]
        else:
            hd = frame_hash - current_root_hash
            time_diff = abs(frame_item["timestamp"] - current_root_ts)

            is_same_cluster = (hd < hd_thresh)
            if is_image_dir and time_diff >= 1.0:
                is_same_cluster = False

            if is_same_cluster:
                current_cluster.append(frame_item)
            else:
                process_and_finalize_cluster(current_cluster)
                current_cluster = [frame_item]
                current_root_hash = frame_hash
                current_root_ts = frame_item["timestamp"]

    if current_cluster:
        process_and_finalize_cluster(current_cluster)

    return survived_items, total_processed_frames, total_clusters


print('✅ Bộ lọc FastDedup (pHash + Object-Density) đã sẵn sàng!')


## 📊 Bước 6: Giai đoạn 2 — Automated Data Profiling & Pre-labeling
- **Day/Night**: V-mean < 50.0 hoặc (V-mean < 65.0 và P90 < 100.0) -> Tránh nhầm đường nhựa tối màu là ban đêm.
- **Blur**: Phương sai Laplacian < 100.0.
- **Occlusion**: Tỷ lệ giao nhau trên vùng nhỏ hơn (IoA) >= 0.30.
- **Small Object**: Diện tích bbox < 0.5% diện tích ảnh.

In [ ]:
# ==============================================================================
# BƯỚC 6: STAGE 2 — AUTOMATED DATA PROFILING & PRE-LABELING
# Kiểm tra: Ngày/Đêm (V-mean + P90), Mờ nhòe (Blur), Che khuất (IoA), Vật thể nhỏ
# ==============================================================================

"""
Stage 2: Automated Data Profiling & Pre-label Generation
Extracts metadata (Day/Night, Blur, Small Object, Occlusion, Density) and exports CVAT pre-labels.
"""

import json
from pathlib import Path
import cv2
import numpy as np
from tqdm import tqdm

from .config import (
    COCO_TO_GREENSM_MAP,
    NIGHT_V_THRESH,
    NIGHT_P90_THRESH,
    NIGHT_EXTREME_THRESH,
    BLUR_THRESH,
    SMALL_OBJECT_RATIO,
    OCCLUSION_IOA_THRESH,
    DENSE_SCENE_COUNT,
)
from .utils import calculate_ioa


def profile_image(survived_item):
    """
    STAGE 2: AUTOMATED DATA PROFILING
    Computes metadata metrics for survived image using cached YOLO detections.
    """
    bgr_img = survived_item["bgr_img"]

    # 1. Day / Night Tag (Robust Multi-criteria)
    hsv = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2HSV)
    v_channel = hsv[:, :, 2]
    brightness_v = float(np.mean(v_channel))
    p90_brightness = float(np.percentile(v_channel, 90))

    # A scene is truly night if overall brightness < 50.0,
    # OR if brightness < 65.0 AND upper ambient light (p90) < 100.0.
    is_night = bool(
        (brightness_v < NIGHT_EXTREME_THRESH) or
        (brightness_v < NIGHT_V_THRESH and p90_brightness < NIGHT_P90_THRESH)
    )

    # 2. Blur Tag (Laplacian variance < 100)
    if "laplacian_var" in survived_item:
        laplacian_var = survived_item["laplacian_var"]
    else:
        gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
        laplacian_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    is_blur = bool(laplacian_var < BLUR_THRESH)

    detections = survived_item.get("detections", [])
    vehicle_count = len(detections)

    # 3. Scale Variance Tag (has_small_object: vehicle box area / img_area < 0.01)
    has_small_object = False
    for det in detections:
        w_norm, h_norm = det["bbox_xywhn"][2], det["bbox_xywhn"][3]
        box_ratio = w_norm * h_norm
        if box_ratio < SMALL_OBJECT_RATIO:
            has_small_object = True
            break

    # 4. High Occlusion Tag (IoA > 0.30 between any pair of vehicle BBoxes)
    has_high_occlusion = False
    n_det = len(detections)
    for i in range(n_det):
        for j in range(n_det):
            if i == j:
                continue
            ioa = calculate_ioa(detections[i]["bbox_xyxy"], detections[j]["bbox_xyxy"])
            if ioa > OCCLUSION_IOA_THRESH:
                has_high_occlusion = True
                break
        if has_high_occlusion:
            break

    # 5. Density Tag (vehicle_count > 12)
    is_dense_scene = bool(vehicle_count > DENSE_SCENE_COUNT)

    profile = {
        "source": survived_item["source"],
        "vehicle_count": vehicle_count,
        "brightness_v": round(brightness_v, 2),
        "laplacian_var": round(laplacian_var, 2),
        "tags": {
            "is_night": is_night,
            "is_blur": is_blur,
            "has_small_object": has_small_object,
            "has_high_occlusion": has_high_occlusion,
            "is_dense_scene": is_dense_scene
        }
    }

    return profile


def export_pre_labels(survived_item, pre_labels_dir):
    """
    Exports pre-labels in YOLO normalized format (class x_center y_center width height)
    to <pre_labels_dir>/<standard_image_stem>.txt mapped to GreenSM 5-class taxonomy:
    (0: greensm, 1: car, 2: motorcycle, 3: bus, 4: truck)
    """
    img_name = survived_item["name"]
    stem = Path(img_name).stem
    txt_path = Path(pre_labels_dir) / f"{stem}.txt"

    lines = []
    for det in survived_item.get("detections", []):
        coco_cls = det["cls"]
        target_cls = COCO_TO_GREENSM_MAP.get(coco_cls, coco_cls)
        x_c, y_c, w, h = det["bbox_xywhn"]
        lines.append(f"{target_cls} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

    with open(txt_path, "w", encoding="utf-8") as f:
        f.writelines(lines)


def run_data_profiling(survived_items, pre_labels_dir, data_profile_json):
    """
    Run Stage 2 profiling and pre-label export for all survived items.
    Updates and saves data_profile.json.
    """
    pre_labels_dir = Path(pre_labels_dir)
    data_profile_json = Path(data_profile_json)

    pre_labels_dir.mkdir(parents=True, exist_ok=True)

    # Load existing profile to merge incremental runs seamlessly
    existing_profile = {}
    if data_profile_json.exists():
        try:
            with open(data_profile_json, "r", encoding="utf-8") as f:
                existing_profile = json.load(f)
        except Exception:
            existing_profile = {}

    for item in tqdm(survived_items, desc="Profiling Survived Images"):
        profile_data = profile_image(item)
        existing_profile[item["name"]] = profile_data
        export_pre_labels(item, pre_labels_dir)

    with open(data_profile_json, "w", encoding="utf-8") as f:
        json.dump(existing_profile, f, indent=2, ensure_ascii=False)

    print(f"[INFO] Updated metadata profile at: {data_profile_json}")
    print(f"[INFO] Pre-labels saved at: {pre_labels_dir}")

    return existing_profile


print('✅ Bộ Profiling & Gán nhãn sơ bộ đã sẵn sàng!')


## 🚀 Bước 7: Thực thi Toàn bộ Pipeline (Run Pipeline)
Quét toàn bộ video và ảnh trong `data/inputs/`, thực hiện Dedup + Profiling và lưu kết quả vào `data/processed_frames`.

In [ ]:
# ==============================================================================
# BƯỚC 7: THỰC THI TOÀN BỘ PIPELINE TRÊN TẬP DỮ LIỆU
# ==============================================================================
import time
from glob import glob
from ultralytics import YOLO

# Khởi tạo mô hình YOLO11n (COCO pretrained)
print("📥 Đang tải mô hình YOLO11n...")
yolo_model = YOLO(Config.YOLO_MODEL_PATH)
print("✅ Đã sẵn sàng YOLO11n.")

# Đường dẫn dữ liệu
video_dir = Config.RAW_VIDEOS_DIR
image_dir = Config.RAW_IMAGES_DIR
output_dir = Config.PROCESSED_FRAMES_DIR
report_dir = Config.REPORTS_DIR

os.makedirs(output_dir, exist_ok=True)
os.makedirs(report_dir, exist_ok=True)

video_files = glob(os.path.join(video_dir, "*.mp4")) + glob(os.path.join(video_dir, "*.avi")) + glob(os.path.join(video_dir, "*.mov"))
image_files = glob(os.path.join(image_dir, "*.jpg")) + glob(os.path.join(image_dir, "*.jpeg")) + glob(os.path.join(image_dir, "*.png"))

print(f"🎬 Tìm thấy: {len(video_files)} video và {len(image_files)} ảnh đầu vào.")

all_metadata = []
start_total = time.time()

# 1. Xử lý Video
for vpath in video_files:
    vname = os.path.basename(vpath)
    vid_id = generate_video_id(vname)
    print("=" * 60)
    print(f"▶️ Bắt đầu xử lý Video: {vname} (Mã định danh: vid_{vid_id})")
    print("=" * 60)
    
    # Stage 1: FastDedup
    selected_frames = run_fast_dedup(vpath, yolo_model, vid_id=vid_id)
    print(f"🎯 Sau Dedup: Giữ lại {len(selected_frames)} frame tối ưu mật độ xe.")
    
    # Stage 2: Profiling
    meta_list = run_profiling(selected_frames, output_dir, yolo_model, source_type="video")
    all_metadata.extend(meta_list)

# 2. Xử lý Ảnh rời rạc
if image_files:
    print("=" * 60)
    print(f"▶️ Bắt đầu xử lý {len(image_files)} ảnh rời rạc...")
    print("=" * 60)
    img_stream = stream_images_dir(image_dir)
    img_meta_list = run_profiling(img_stream, output_dir, yolo_model, source_type="image")
    all_metadata.extend(img_meta_list)

# Lưu báo cáo Profiling JSON
profiling_report_path = os.path.join(report_dir, "profiling_report.json")
with open(profiling_report_path, "w", encoding="utf-8") as f:
    json.dump(all_metadata, f, indent=2, ensure_ascii=False)

elapsed = time.time() - start_total
print("=" * 60)
print(f"🎉 HOÀN THÀNH TOÀN BỘ PIPELINE trong {elapsed:.2f}s!")
print(f"📊 Tổng số frame/ảnh chuẩn đã lưu: {len(all_metadata)}")
print(f"📄 Báo cáo profiling lưu tại: {profiling_report_path}")


## 📈 Bước 8: Trực quan hóa Thống kê Metadata Profiling
Phân tích phân bố dữ liệu: Ngày/Đêm, Mờ/Rõ, Mật độ xe, Che khuất.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

report_file = os.path.join(Config.REPORTS_DIR, 'profiling_report.json')
if os.path.exists(report_file):
    with open(report_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f'📊 Tổng quan tập dữ liệu: {len(df)} ảnh')
    print(df[['source_type', 'is_night', 'is_blur', 'num_vehicles', 'has_occlusion', 'has_small_objects']].describe())
    
    # Biểu đồ phân bố
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    df['is_night'].value_counts().plot(kind='bar', ax=axes[0], color=['orange', 'navy'])
    axes[0].set_title('Phân bố Ngày (False) / Đêm (True)')
    axes[0].set_ylabel('Số lượng ảnh')
    
    df['num_vehicles'].plot(kind='hist', bins=15, ax=axes[1], color='teal', edgecolor='black')
    axes[1].set_title('Phân bố Mật độ Xe / Ảnh')
    axes[1].set_xlabel('Số lượng xe')
    
    df['is_blur'].value_counts().plot(kind='bar', ax=axes[2], color=['green', 'red'])
    axes[2].set_title('Phân bố Rõ (False) / Mờ (True)')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ Chưa có file profiling_report.json. Vui lòng chạy Bước 7 trước!')

## 📦 Bước 9: Đóng gói Chuẩn CVAT (Native XML 1.1 + Split Zip 50 frames)
Tự động tạo `annotations.xml` chuẩn CVAT 1.1 và chia thành các file zip nhỏ (50 frame/zip).
- Import siêu tốc (< 1s cho file XML ~28KB)
- Giữ nguyên tên file không bị mất liên kết.

In [ ]:
# ==============================================================================
# BƯỚC 9: ĐÓNG GÓI CHUẨN CVAT (Native XML 1.1 + Split Zip 50 frames)
# ==============================================================================

"""
CVAT Packaging Module: Generates Native XML (CVAT 1.1) and Split YOLO ZIP archives
"""

import json
import zipfile
from pathlib import Path
from datetime import datetime
from PIL import Image

from .config import (
    GREENSM_CLASSES,
    CLASS_COLORS,
    DEFAULT_BATCH_SIZE,
    DEFAULT_OUTPUT_DIR,
)


def load_manifest(manifest_path):
    """Load manifest tracking file for previously packaged items."""
    manifest_path = Path(manifest_path)
    if manifest_path.exists():
        try:
            with open(manifest_path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return {"zipped_files": [], "batches": []}


def save_manifest(manifest_path, manifest_data):
    """Save manifest tracking file."""
    manifest_path = Path(manifest_path)
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=2, ensure_ascii=False)


def create_cvat_xml_annotations(images_list, dedup_images_dir, pre_labels_dir, xml_path):
    """
    Generate native CVAT 1.1 / CVAT for images 1.1 XML annotation file.
    This format is natively supported by CVAT and avoids all path matching issues of YOLO.
    """
    dedup_images_dir = Path(dedup_images_dir)
    pre_labels_dir = Path(pre_labels_dir)
    xml_path = Path(xml_path)

    xml_lines = [
        '<?xml version="1.0" encoding="utf-8"?>',
        '<annotations>',
        '  <version>1.1</version>',
        '  <meta>',
        '    <task>',
        '      <id>1</id>',
        '      <name>GreenSM_PreLabels</name>',
        f'      <size>{len(images_list)}</size>',
        '      <mode>annotation</mode>',
        '      <labels>'
    ]

    for c in GREENSM_CLASSES:
        color = CLASS_COLORS.get(c, "#FFFFFF")
        xml_lines.append(f'        <label><name>{c}</name><color>{color}</color></label>')

    xml_lines.extend([
        '      </labels>',
        '    </task>',
        '  </meta>'
    ])

    total_boxes = 0

    for idx, img_file in enumerate(images_list):
        img_name = img_file.name
        img_path = dedup_images_dir / img_name

        try:
            with Image.open(img_path) as im:
                w, h = im.size
        except Exception:
            w, h = 1920, 1080

        xml_lines.append(f'  <image id="{idx}" name="{img_name}" width="{w}" height="{h}">')

        txt_name = f"{img_file.stem}.txt"
        txt_path = pre_labels_dir / txt_name

        if txt_path.exists():
            with open(txt_path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        try:
                            cls_id = int(parts[0])
                            xc, yc, bw, bh = map(float, parts[1:5])
                            label_name = GREENSM_CLASSES[cls_id] if cls_id < len(GREENSM_CLASSES) else "car"

                            xtl = max(0.0, (xc - bw / 2.0) * w)
                            ytl = max(0.0, (yc - bh / 2.0) * h)
                            xbr = min(float(w), (xc + bw / 2.0) * w)
                            ybr = min(float(h), (yc + bh / 2.0) * h)

                            xml_lines.append(
                                f'    <box label="{label_name}" occluded="0" xtl="{xtl:.2f}" ytl="{ytl:.2f}" '
                                f'xbr="{xbr:.2f}" ybr="{ybr:.2f}" z_order="0">\n    </box>'
                            )
                            total_boxes += 1
                        except Exception:
                            continue

        xml_lines.append('  </image>')

    xml_lines.append('</annotations>')

    with open(xml_path, "w", encoding="utf-8") as f:
        f.write("\n".join(xml_lines))

    return total_boxes


def create_single_zip_batch(zip_path, images_chunk, pre_labels_dir, include_images=True):
    """Helper to create a single YOLO 1.1 zip file with exact filename matching."""
    zip_path = Path(zip_path)
    pre_labels_dir = Path(pre_labels_dir)

    obj_names_content = "\n".join(GREENSM_CLASSES) + "\n"
    obj_data_content = (
        f"classes = {len(GREENSM_CLASSES)}\n"
        f"train = train.txt\n"
        f"names = obj.names\n"
        f"backup = backup/\n"
    )

    train_txt_lines = []

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("obj.names", obj_names_content)
        zf.writestr("obj.data", obj_data_content)

        for img_file in images_chunk:
            stem = img_file.stem
            img_name = img_file.name
            txt_name = f"{stem}.txt"

            train_txt_lines.append(f"{img_name}\n")
            train_txt_lines.append(f"obj_train_data/{img_name}\n")

            if include_images and img_file.exists():
                zf.write(img_file, arcname=f"obj_train_data/{img_name}")
                zf.write(img_file, arcname=img_name)

            txt_file = pre_labels_dir / txt_name
            if txt_file.exists():
                txt_data = txt_file.read_bytes()
            else:
                txt_data = b""

            zf.writestr(f"obj_train_data/{txt_name}", txt_data)
            zf.writestr(txt_name, txt_data)

        zf.writestr("train.txt", "".join(train_txt_lines))


def create_cvat_zip_package(output_dir=DEFAULT_OUTPUT_DIR, zip_dir=None, batch_size=DEFAULT_BATCH_SIZE, force_all=False):
    """
    Creates CVAT Native XML packages and split YOLO batches.
    """
    output_dir = Path(output_dir).resolve()
    zip_dir = Path(zip_dir).resolve() if zip_dir else output_dir / "cvat_uploads"

    dedup_images_dir = output_dir / "deduplicated_images"
    pre_labels_dir = output_dir / "pre_labels"
    manifest_path = output_dir / ".zipped_manifest.json"

    if not dedup_images_dir.exists():
        print(f"[ERROR] Deduplicated images directory does not exist: {dedup_images_dir}")
        return None

    manifest = load_manifest(manifest_path)
    already_zipped = set(manifest.get("zipped_files", [])) if not force_all else set()

    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff', '.tif'}
    all_images = sorted([
        f for f in dedup_images_dir.iterdir()
        if f.is_file() and f.suffix.lower() in image_extensions
    ])

    if not all_images:
        print(f"[INFO] No images found in {dedup_images_dir}.")
        return None

    new_images = [img for img in all_images if img.name not in already_zipped]

    if not new_images:
        print(f"[INFO] All {len(all_images)} image(s) in '{dedup_images_dir.name}' have ALREADY been packaged in previous ZIP batch(es).")
        print(f"       No new un-zipped images found. Skipping packaging to prevent duplication.")
        return None

    print(f"[INFO] Found {len(new_images)} NEW un-packaged image(s) out of {len(all_images)} total.")
    zip_dir.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

    # 1. Master CVAT Native XML (ALL Images)
    master_xml_filename = f"cvat_annotations_ALL_{timestamp_str}.xml"
    master_xml_path = zip_dir / master_xml_filename
    master_xml_zip_filename = f"cvat_NATIVE_XML_ALL_{timestamp_str}.zip"
    master_xml_zip_path = zip_dir / master_xml_zip_filename

    total_boxes = create_cvat_xml_annotations(new_images, dedup_images_dir, pre_labels_dir, master_xml_path)
    with zipfile.ZipFile(master_xml_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(master_xml_path, arcname="annotations.xml")

    # 2. Split Batches (50 images/part)
    chunks = [new_images[i:i + batch_size] for i in range(0, len(new_images), batch_size)]
    total_chunks = len(chunks)
    packaged_names = []

    print(f"\n=== GENERATING {total_chunks} SPLIT BATCHES ({batch_size} IMAGES/PART) ===")
    for idx, chunk in enumerate(chunks, start=1):
        # Image + YOLO zip
        zip_filename = f"cvat_batch_{timestamp_str}_part{idx:02d}_of_{total_chunks:02d}.zip"
        zip_path = zip_dir / zip_filename
        create_single_zip_batch(zip_path, chunk, pre_labels_dir, include_images=True)

        # Native XML zip for this part
        part_xml_filename = f"cvat_NATIVE_XML_part{idx:02d}_of_{total_chunks:02d}.xml"
        part_xml_path = zip_dir / part_xml_filename
        part_xml_zip_path = zip_dir / f"cvat_NATIVE_XML_part{idx:02d}_of_{total_chunks:02d}.zip"

        create_cvat_xml_annotations(chunk, dedup_images_dir, pre_labels_dir, part_xml_path)
        with zipfile.ZipFile(part_xml_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(part_xml_path, arcname="annotations.xml")

        print(f"[INFO] Part [{idx}/{total_chunks}] ({len(chunk)} images):")
        print(f"       - Image ZIP: {zip_filename}")
        print(f"       - Native XML ZIP: {part_xml_zip_path.name}")

        for img in chunk:
            packaged_names.append(img.name)

    manifest["zipped_files"].extend(packaged_names)
    manifest["batches"].append({
        "timestamp": datetime.now().isoformat(),
        "image_count": len(packaged_names),
        "total_boxes": total_boxes,
        "master_xml": master_xml_zip_filename
    })
    save_manifest(manifest_path, manifest)

    print(f"\n[SUCCESS] Packaging Completed!")
    print(f"  ⭐ Dùng file XML này nếu Task chứa TOÀN BỘ {len(new_images)} ảnh: {master_xml_zip_path.name}")
    print(f"  ⭐ Dùng file XML part tương ứng nếu bạn tạo Task theo từng Part:")
    for idx in range(1, total_chunks + 1):
        print(f"     - Task Part {idx}: cvat_NATIVE_XML_part{idx:02d}_of_{total_chunks:02d}.zip")

    return master_xml_zip_path


print("📦 Bắt đầu đóng gói cho CVAT...")
create_cvat_package(
    processed_dir=Config.PROCESSED_FRAMES_DIR,
    report_json_path=os.path.join(Config.REPORTS_DIR, "profiling_report.json"),
    output_zip_dir=Config.CVAT_READY_DIR,
    frames_per_zip=50
)


## 💾 Bước 10: Tải Kết quả về Máy tính (Download to Local)
Tải các file zip CVAT và báo cáo profiling trực tiếp về máy tính của bạn.

In [ ]:
from google.colab import files
from glob import glob

zip_files = glob(os.path.join(Config.CVAT_READY_DIR, '*.zip'))
print(f'📦 Tìm thấy {len(zip_files)} file zip CVAT sẵn sàng tải về:')
for zf in zip_files:
    print(f' - {zf}')
    # Bỏ comment dòng dưới nếu muốn tự động tải về trình duyệt:
    # files.download(zf)

# Tải file báo cáo JSON
report_path = os.path.join(Config.REPORTS_DIR, 'profiling_report.json')
if os.path.exists(report_path):
    print(f'📄 Tải file báo cáo: {report_path}')
    # files.download(report_path)